In [39]:
import torch.nn as nn
import torch
seed = 42
torch.manual_seed(seed)

假如我要对一个fashionmnist试用gan，那大小应该是28*28=784.
damn,我搞错了，gan不需要encoder和decoder，只用generator和discriminator

In [40]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main=nn.Sequential(
            nn.Linear(100, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, 784),
            nn.Tanh()#稳定输出范围在-1到1之间。最后生成的图像和真实的在数值范围上会相同，这样判断真假就不会从数字大小入手
            #Tanh可以比sigmoid更有效的传播梯度
        )
    def forward(self, input):
        output=self.main(input)
        output=output.view(-1, 1, 28, 28)
        return output

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main=nn.Sequential(
            nn.Linear(784, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, input):
        input=input.view(input.shape[0],-1)
        output=self.main(input)
        return output

LeakyReLU输出负数时不输出0，而是一个绝对值很小的负数

In [10]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [58]:
import torchvision
Train_dataset=torchvision.datasets.CIFAR10(root=".",download=True,transform=torchvision.transforms.ToTensor(),train=True)
Test_dataset=torchvision.datasets.CIFAR10(root=".",download=True,transform=torchvision.transforms.ToTensor(),train=False)
train_loader=torch.utils.data.DataLoader(Train_dataset, batch_size=256, shuffle=True)
test_loader=torch.utils.data.DataLoader(Test_dataset, batch_size=256, shuffle=False)

In [66]:
generator=Generator().to(device)
discriminator=Discriminator().to(device)
model=(generator,discriminator)
weight_decay = 8e-9
beta1 = 0.5
beta2 = 0.999
optimizer_G=torch.optim.Adam(generator.parameters(), lr=0.0002,weight_decay=weight_decay,betas=(beta1,beta2))
optimizer_D=torch.optim.Adam(discriminator.parameters(), lr=0.0002,weight_decay=weight_decay,betas=(beta1,beta2))
#分别设置优化器
optimizer=(optimizer_G,optimizer_D)


In [64]:
import matplotlib.pyplot as plt
def train(model, optimizer, train_loader, epochs,device):
    generator, discriminator=model
    optimizer_G, optimizer_D=optimizer
    for epoch in range(epochs):
        #先训练生成器
        generator.train()
        discriminator.train()

        for idx,(datas,label) in enumerate(train_loader):
            discriminator.eval()

            datas=datas.to(device)
            #先要生成一批向量，然后用generator生成图片
            noise = torch.randn(datas.shape[0], 100).to(device)
            real_label=torch.ones(datas.shape[0], 1).to(device)
            fake_label=torch.zeros(datas.shape[0], 1).to(device)
            predict=generator(noise)
            # print(predict.shape)
            optimizer_G.zero_grad()
            disc_pre=discriminator(predict)
            #要算判别器判断图片为真的概率，算与真图片的概率，loss算差，提高disc_pre与1的接近程度
            loss_gn=torch.nn.functional.binary_cross_entropy(disc_pre, real_label)
            loss_gn.backward()
            optimizer_G.step()
            

            discriminator.train()
            disc_real=discriminator(datas)
            real_loss=torch.nn.functional.binary_cross_entropy(disc_real, real_label)
            disc_fake=discriminator(predict.detach())#判别器判别一个假的数据为真的概率
            #假如detach从计算图剥离，不刘翔前面的梯度
            optimizer_D.zero_grad()
            fake_loss=torch.nn.functional.binary_cross_entropy(disc_fake, fake_label)
            loss_total=(real_loss+fake_loss)/2.0
            loss_total.backward()
            optimizer_D.step()
            

            if idx%100==0:
                print(f"Epoch: {epoch},生成器损失: {loss_gn.item()} ,判别器损失: {loss_total.item()}")
                #测试生成器图片
                generator.eval()
                noise=torch.randn(10, 100).to(device)
                imgs=generator(noise)
                print(imgs.shape)
                figure,axs=plt.subplots(1, 10)
                for i in range(10):
                    axs[i].imshow(imgs[i].cpu().detach().numpy().reshape(28,28))#不能向axs传递1，28，28
                plt.show()

In [ ]:
train(model,optimizer,train_loader,50,device)

DCGgan

In [ ]:
class DCGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.ConvTranspose2d(100, 256, kernel_size=(3, 3), stride=(2, 2), bias=False)
        self.bn1 = nn.BatchNorm2d(256)
        self.conv2 = nn.ConvTranspose2d(256, 128, kernel_size=(3, 3), stride=(2, 2), bias=False)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.ConvTranspose2d(128, 64, kernel_size=(3, 3), stride=(2, 2), bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.ConvTranspose2d(64, 1, kernel_size=(3, 3), stride=(2, 2), padding=(2, 2), output_padding=(1, 1), bias=False)
        self.tanh = nn.Tanh()
        self.relu = nn.ReLU()

    def forward(self, input):
        hidden1 = self.relu(self.bn1(self.conv1(input)))
        hidden2 = self.relu(self.bn2(self.conv2(hidden1)))
        hidden3 = self.relu(self.bn3(self.conv3(hidden2)))
        generated = self.tanh(self.conv4(hidden3)).view(input.shape[0], 1, 28, 28)
        return generated